# SciCode-Bench: Formula 1 Race Strategy Optimization

## Overview

Formula 1 race strategy is a rich engineering optimization problem. Teams must balance tire degradation, fuel load effects, pit stop timing, and compound selection to minimize total race time. This benchmark evaluates a model's ability to implement physical degradation models, perform single-variable optimization, and extend to multi-variable sensitivity analysis.

## Race Parameters (use these exact values throughout all subtasks)

| Parameter | Value |
|---|---|
| Total race laps | 57 |
| Starting fuel load (kg) | 110 |
| Fuel burn per lap (kg) | 1.85 |
| Fuel effect on lap time (s/kg) | 0.035 |
| Pit stop time loss (s) | 22.0 |

## Tire Compound Parameters

| Compound | Base Lap Time (s) | Degradation Rate |
|---|---|---|
| Soft | 90.5 | 0.080 |
| Medium | 91.2 | 0.045 |
| Hard | 92.0 | 0.022 |

---

## Lap Time Model

The lap time at a given lap number on a given tire compound is modeled as:

```
lap_time(lap_on_tire, fuel_remaining) = base_time + degradation_rate × lap_on_tire² + fuel_effect × fuel_remaining
```

Where:
- `lap_on_tire` is the number of laps completed on the **current** set of tires (starts at 0)
- `fuel_remaining` is the fuel remaining at that point in the race
- `fuel_remaining(race_lap) = starting_fuel - fuel_burn_per_lap × race_lap`

---

In [ ]:
# Install dependencies
!pip install scipy numpy --quiet

In [ ]:
import numpy as np
import unittest
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ── Global Race Constants ──────────────────────────────────────────────────────
TOTAL_LAPS       = 57
STARTING_FUEL    = 110.0   # kg
FUEL_BURN        = 1.85    # kg per lap
FUEL_EFFECT      = 0.035   # seconds per kg
PIT_STOP_LOSS    = 22.0    # seconds

COMPOUNDS = {
    'Soft':   {'base_time': 90.5, 'deg_rate': 0.080},
    'Medium': {'base_time': 91.2, 'deg_rate': 0.045},
    'Hard':   {'base_time': 92.0, 'deg_rate': 0.022},
}

print("Constants loaded successfully.")

---
# Subtask 1: Tire Degradation & Lap Time Modeling

## Problem

Implement a function `compute_lap_time(compound, lap_on_tire, race_lap)` that returns the predicted lap time in seconds for a given tire compound at a specific point in the race.

### Inputs
- `compound` (str): One of `'Soft'`, `'Medium'`, `'Hard'`
- `lap_on_tire` (int): Number of laps already completed on this tire set (0-indexed, so first lap on fresh tires = 0)
- `race_lap` (int): Current race lap number (0-indexed, so first lap of race = 0)

### Output
- A single `float`: the predicted lap time in seconds

### Formula
```
fuel_remaining  = STARTING_FUEL - FUEL_BURN × race_lap
lap_time        = base_time + deg_rate × lap_on_tire² + FUEL_EFFECT × fuel_remaining
```

### Example
For `compound='Soft'`, `lap_on_tire=0`, `race_lap=0`:
```
fuel_remaining = 110.0 - 1.85 × 0 = 110.0
lap_time = 90.5 + 0.080 × 0² + 0.035 × 110.0 = 90.5 + 0 + 3.85 = 94.35
```
Expected output: **94.35**

In [ ]:
# ── SUBTASK 1: Implement your solution here ────────────────────────────────────

def compute_lap_time(compound: str, lap_on_tire: int, race_lap: int) -> float:
    """
    Compute predicted lap time for a given compound and race situation.

    Parameters
    ----------
    compound    : 'Soft', 'Medium', or 'Hard'
    lap_on_tire : laps completed on this tire set (0 = fresh)
    race_lap    : current lap of the race (0 = lap 1)

    Returns
    -------
    float : predicted lap time in seconds
    """
    # YOUR CODE HERE
    pass


# Quick smoke test
result = compute_lap_time('Soft', 0, 0)
print(f"Subtask 1 smoke test — Soft, lap_on_tire=0, race_lap=0: {result}")
print(f"Expected: 94.35")

## Unit Tests — Subtask 1

In [ ]:
class TestSubtask1(unittest.TestCase):

    # ── Output existence ──────────────────────────────────────────────────────
    def test_output_exists(self):
        """Function must return a non-None value."""
        result = compute_lap_time('Medium', 5, 5)
        self.assertIsNotNone(result)

    def test_output_is_float(self):
        """Return type must be numeric."""
        result = compute_lap_time('Hard', 10, 10)
        self.assertIsInstance(result, (int, float))

    def test_output_is_positive(self):
        """Lap time must be a positive number."""
        for compound in ['Soft', 'Medium', 'Hard']:
            self.assertGreater(compute_lap_time(compound, 0, 0), 0)

    # ── Output correctness ────────────────────────────────────────────────────
    def test_soft_fresh_lap1(self):
        """Soft tires, fresh, lap 1 of race."""
        self.assertAlmostEqual(compute_lap_time('Soft', 0, 0), 94.35, places=3)

    def test_medium_lap10_race10(self):
        """Medium tires, 10 laps on tire, race lap 10."""
        # fuel_remaining = 110 - 1.85*10 = 91.5
        # lap_time = 91.2 + 0.045*(10^2) + 0.035*91.5 = 91.2 + 4.5 + 3.2025 = 98.9025
        self.assertAlmostEqual(compute_lap_time('Medium', 10, 10), 98.9025, places=3)

    def test_hard_lap20_race40(self):
        """Hard tires, 20 laps on tire, race lap 40."""
        # fuel_remaining = 110 - 1.85*40 = 36.0
        # lap_time = 92.0 + 0.022*(20^2) + 0.035*36.0 = 92.0 + 8.8 + 1.26 = 102.06
        self.assertAlmostEqual(compute_lap_time('Hard', 20, 40), 102.06, places=3)

    # ── Boundary checks ───────────────────────────────────────────────────────
    def test_fuel_decreases_lap_time(self):
        """Later in race (less fuel) → shorter lap time, all else equal."""
        early = compute_lap_time('Medium', 5, 5)
        late  = compute_lap_time('Medium', 5, 45)
        self.assertGreater(early, late)

    def test_degradation_increases_lap_time(self):
        """More laps on tire → longer lap time, all else equal."""
        fresh = compute_lap_time('Soft', 0, 10)
        worn  = compute_lap_time('Soft', 20, 10)
        self.assertGreater(worn, fresh)

    def test_soft_faster_than_hard_when_fresh(self):
        """Soft tires must be faster than Hard when both are fresh."""
        soft = compute_lap_time('Soft', 0, 0)
        hard = compute_lap_time('Hard', 0, 0)
        self.assertLess(soft, hard)

    def test_soft_slower_than_hard_when_very_worn(self):
        """Highly degraded Soft tires must be slower than equally worn Hard tires."""
        soft_worn = compute_lap_time('Soft', 35, 35)
        hard_worn = compute_lap_time('Hard', 35, 35)
        self.assertGreater(soft_worn, hard_worn)

    def test_all_compounds_accepted(self):
        """Function must accept all three valid compound names."""
        for compound in ['Soft', 'Medium', 'Hard']:
            result = compute_lap_time(compound, 5, 5)
            self.assertIsNotNone(result)


suite1 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask1)
runner = unittest.TextTestRunner(verbosity=2)
print("\n" + "="*60)
print("SUBTASK 1 UNIT TESTS")
print("="*60)
runner.run(suite1)

---
# Subtask 2: Optimal One-Stop Pit Window

## Problem

Using your `compute_lap_time` function from Subtask 1, implement `optimize_one_stop(stint1_compound, stint2_compound)` that finds the optimal lap to pit (for a single pit stop strategy) that minimizes total race time.

### Strategy Structure
- **Stint 1:** Laps 1 → pit_lap on `stint1_compound`
- **Pit stop:** +22.0 seconds added once
- **Stint 2:** Remaining laps on `stint2_compound` (fresh tires)

### Inputs
- `stint1_compound` (str): Tire compound for first stint
- `stint2_compound` (str): Tire compound for second stint

### Output
A tuple `(optimal_pit_lap, total_race_time)` where:
- `optimal_pit_lap` (int): Race lap number after which the pit stop occurs (valid range: lap 10 to lap 46)
- `total_race_time` (float): Total race time in seconds for the optimal strategy

### Notes
- Search over all integer pit laps from 10 to 46 inclusive
- `lap_on_tire` resets to 0 at the start of each stint
- `race_lap` always tracks the absolute lap number in the race (0-indexed)
- The pit stop time loss is added once to the total

### Expected Output
For `stint1_compound='Soft'`, `stint2_compound='Hard'`:
- `optimal_pit_lap` = **18**
- `total_race_time` ≈ **5594.3 seconds** (within ±1.0s)

In [ ]:
# ── SUBTASK 2: Implement your solution here ────────────────────────────────────

def optimize_one_stop(stint1_compound: str, stint2_compound: str) -> tuple:
    """
    Find the optimal pit lap for a one-stop strategy.

    Parameters
    ----------
    stint1_compound : tire compound for the first stint
    stint2_compound : tire compound for the second stint

    Returns
    -------
    (optimal_pit_lap: int, total_race_time: float)
    """
    # YOUR CODE HERE
    pass


# Quick smoke test
pit_lap, race_time = optimize_one_stop('Soft', 'Hard')
print(f"Subtask 2 smoke test — Soft → Hard strategy:")
print(f"  Optimal pit lap : {pit_lap}  (expected: 18)")
print(f"  Total race time : {race_time:.3f}s  (expected: ~5594.3s)")

## Unit Tests — Subtask 2

In [ ]:
class TestSubtask2(unittest.TestCase):

    # ── Output existence ──────────────────────────────────────────────────────
    def test_output_exists(self):
        result = optimize_one_stop('Soft', 'Hard')
        self.assertIsNotNone(result)

    def test_output_is_tuple(self):
        result = optimize_one_stop('Soft', 'Hard')
        self.assertIsInstance(result, tuple)
        self.assertEqual(len(result), 2)

    def test_output_types(self):
        pit_lap, race_time = optimize_one_stop('Medium', 'Hard')
        self.assertIsInstance(pit_lap, (int, np.integer))
        self.assertIsInstance(race_time, (int, float, np.floating))

    # ── Output correctness ────────────────────────────────────────────────────
    def test_soft_hard_optimal_pit_lap(self):
        """Soft → Hard: optimal pit lap must be 18."""
        pit_lap, _ = optimize_one_stop('Soft', 'Hard')
        self.assertEqual(int(pit_lap), 18)

    def test_soft_hard_total_race_time(self):
        """Soft → Hard: total race time must be within 1.0s of expected."""
        _, race_time = optimize_one_stop('Soft', 'Hard')
        self.assertAlmostEqual(race_time, 5594.3, delta=1.0)

    def test_medium_hard_optimal_pit_lap(self):
        """Medium → Hard: optimal pit lap must be 24."""
        pit_lap, _ = optimize_one_stop('Medium', 'Hard')
        self.assertEqual(int(pit_lap), 24)

    # ── Boundary and sanity checks ────────────────────────────────────────────
    def test_pit_lap_in_valid_range(self):
        """Optimal pit lap must be between 10 and 46 inclusive."""
        for c1, c2 in [('Soft', 'Hard'), ('Medium', 'Hard'), ('Soft', 'Medium')]:
            pit_lap, _ = optimize_one_stop(c1, c2)
            self.assertGreaterEqual(int(pit_lap), 10)
            self.assertLessEqual(int(pit_lap), 46)

    def test_one_stop_faster_than_no_stop(self):
        """One-stop strategy must be faster than running entire race on Soft tires."""
        _, one_stop_time = optimize_one_stop('Soft', 'Hard')
        # Compute no-stop time on Soft tires
        no_stop_time = sum(compute_lap_time('Soft', i, i) for i in range(TOTAL_LAPS))
        self.assertLess(one_stop_time, no_stop_time)

    def test_higher_deg_rate_shifts_pit_earlier(self):
        """Softer compound (higher degradation) → earlier optimal pit lap."""
        pit_soft, _   = optimize_one_stop('Soft', 'Hard')
        pit_medium, _ = optimize_one_stop('Medium', 'Hard')
        self.assertLess(int(pit_soft), int(pit_medium))

    def test_race_time_is_positive(self):
        _, race_time = optimize_one_stop('Soft', 'Hard')
        self.assertGreater(race_time, 0)


suite2 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask2)
print("\n" + "="*60)
print("SUBTASK 2 UNIT TESTS")
print("="*60)
runner.run(suite2)

---
# Subtask 3: Two-Stop vs One-Stop Strategy + Sensitivity Analysis

## Problem

Implement `optimize_two_stop(s1_compound, s2_compound, s3_compound)` that finds the optimal two-stop strategy, compares it to the best one-stop, and then performs a sensitivity analysis to find the crossover degradation rate at which the two-stop strategy becomes faster.

### Strategy Structure
- **Stint 1:** Laps 0 → pit1_lap on `s1_compound`
- **Pit stop 1:** +22.0 seconds
- **Stint 2:** pit1_lap → pit2_lap on `s2_compound` (fresh)
- **Pit stop 2:** +22.0 seconds
- **Stint 3:** pit2_lap → end on `s3_compound` (fresh)

### Part A — Two-Stop Optimization
Search over all valid integer combinations of `(pit1_lap, pit2_lap)` where:
- `10 ≤ pit1_lap ≤ 36`
- `pit1_lap + 10 ≤ pit2_lap ≤ 46`
(Minimum 10 laps per stint)

### Part B — Sensitivity Analysis
For the Soft compound only, vary its `deg_rate` from 0.010 to 0.150 in steps of 0.001. At each deg_rate, compute both the best one-stop (Soft→Hard) and best two-stop (Soft→Medium→Hard) total race times. Find the crossover deg_rate where two-stop first becomes faster than one-stop.

### Output
A dictionary with keys:
```python
{
    'pit1_lap'         : int,    # first pit lap
    'pit2_lap'         : int,    # second pit lap
    'two_stop_time'    : float,  # total two-stop race time (seconds)
    'one_stop_time'    : float,  # total one-stop race time (seconds, Soft→Hard)
    'two_stop_wins'    : bool,   # True if two-stop is faster
    'crossover_deg_rate': float  # Soft deg_rate where two-stop first beats one-stop
}
```

### Expected Output
For `s1='Soft'`, `s2='Medium'`, `s3='Hard'`:
- `pit1_lap` = **14**, `pit2_lap` = **34**
- `two_stop_wins` = **False** (at default deg_rates, one-stop is faster)
- `crossover_deg_rate` ≈ **0.095** (within ±0.005)

In [ ]:
# ── SUBTASK 3: Implement your solution here ────────────────────────────────────

def optimize_two_stop(s1_compound: str, s2_compound: str, s3_compound: str) -> dict:
    """
    Find the optimal two-stop strategy and compare to the best one-stop.
    Also find the crossover degradation rate for s1_compound.

    Parameters
    ----------
    s1_compound : first stint compound
    s2_compound : second stint compound
    s3_compound : third stint compound

    Returns
    -------
    dict with keys: pit1_lap, pit2_lap, two_stop_time, one_stop_time,
                    two_stop_wins, crossover_deg_rate
    """
    # YOUR CODE HERE
    pass


# Quick smoke test
result = optimize_two_stop('Soft', 'Medium', 'Hard')
print("Subtask 3 smoke test — Soft → Medium → Hard:")
for k, v in result.items():
    print(f"  {k}: {v}")

## Unit Tests — Subtask 3

In [ ]:
class TestSubtask3(unittest.TestCase):

    @classmethod
    def setUpClass(cls):
        cls.result = optimize_two_stop('Soft', 'Medium', 'Hard')

    # ── Output existence ──────────────────────────────────────────────────────
    def test_output_exists(self):
        self.assertIsNotNone(self.result)

    def test_output_is_dict(self):
        self.assertIsInstance(self.result, dict)

    def test_all_keys_present(self):
        required_keys = {'pit1_lap', 'pit2_lap', 'two_stop_time',
                         'one_stop_time', 'two_stop_wins', 'crossover_deg_rate'}
        self.assertEqual(required_keys, set(self.result.keys()))

    # ── Output correctness ────────────────────────────────────────────────────
    def test_pit1_lap_correct(self):
        self.assertEqual(int(self.result['pit1_lap']), 14)

    def test_pit2_lap_correct(self):
        self.assertEqual(int(self.result['pit2_lap']), 34)

    def test_two_stop_wins_false_at_default_params(self):
        """At default deg_rates, one-stop should be faster."""
        self.assertFalse(self.result['two_stop_wins'])

    def test_crossover_deg_rate(self):
        """Crossover deg_rate must be approximately 0.095."""
        self.assertAlmostEqual(self.result['crossover_deg_rate'], 0.095, delta=0.005)

    def test_two_stop_time_consistent_with_wins_flag(self):
        """two_stop_wins must be consistent with the actual time comparison."""
        wins = self.result['two_stop_time'] < self.result['one_stop_time']
        self.assertEqual(wins, self.result['two_stop_wins'])

    # ── Boundary and sanity checks ────────────────────────────────────────────
    def test_pit_laps_in_valid_range(self):
        pit1 = int(self.result['pit1_lap'])
        pit2 = int(self.result['pit2_lap'])
        self.assertGreaterEqual(pit1, 10)
        self.assertLessEqual(pit1, 36)
        self.assertGreaterEqual(pit2, pit1 + 10)
        self.assertLessEqual(pit2, 46)

    def test_pit_laps_ascending(self):
        self.assertLess(int(self.result['pit1_lap']), int(self.result['pit2_lap']))

    def test_high_degradation_two_stop_wins(self):
        """At very high deg_rate, two-stop must win."""
        original = COMPOUNDS['Soft']['deg_rate']
        COMPOUNDS['Soft']['deg_rate'] = 0.200  # very high
        result_high = optimize_two_stop('Soft', 'Medium', 'Hard')
        COMPOUNDS['Soft']['deg_rate'] = original  # restore
        self.assertTrue(result_high['two_stop_wins'])

    def test_low_degradation_one_stop_wins(self):
        """At very low deg_rate, one-stop must win."""
        original = COMPOUNDS['Soft']['deg_rate']
        COMPOUNDS['Soft']['deg_rate'] = 0.005  # very low
        result_low = optimize_two_stop('Soft', 'Medium', 'Hard')
        COMPOUNDS['Soft']['deg_rate'] = original  # restore
        self.assertFalse(result_low['two_stop_wins'])

    def test_crossover_in_realistic_range(self):
        """Crossover deg_rate must be physically realistic (between 0.01 and 0.15)."""
        crossover = self.result['crossover_deg_rate']
        self.assertGreaterEqual(crossover, 0.010)
        self.assertLessEqual(crossover, 0.150)

    def test_race_times_positive(self):
        self.assertGreater(self.result['two_stop_time'], 0)
        self.assertGreater(self.result['one_stop_time'], 0)


suite3 = unittest.TestLoader().loadTestsFromTestCase(TestSubtask3)
print("\n" + "="*60)
print("SUBTASK 3 UNIT TESTS")
print("="*60)
runner.run(suite3)

---
## How to Run This Benchmark

1. Open this notebook in **Google Colab Pro**
2. Enable **Gemini 3 Pro** via Tools → Gemini in Colab
3. Run the **install-deps** cell first
4. Use Gemini to implement each subtask function (the cells marked `# YOUR CODE HERE`)
5. Run all unit test cells — all tests should pass
6. Use **Runtime → Restart and Run All** to verify the notebook runs end-to-end

**Scoring:** Each subtask is worth 1 point. A subtask is considered solved if all its unit tests pass.